# ATTEST — Performance Benchmark

**ATTEST**: Accumulator-based Tamarin-verified Trust for crEdential Systems in IoT  
Built on **COMPASS** (INDOCRYPT 2025) — a lattice-based accumulator for post-quantum IoT credential revocation.

This notebook reproduces the performance numbers from §5 of the ATTEST paper.
It runs on **Google Colab** (free tier) and takes roughly **5–10 minutes** with set2 (the paper parameter set).

### What is measured
| Section | Operations |
|---------|------------|
| §1 | Component micro-benchmarks (BF ops, ring arithmetic) |
| §2 | `VerifyWithRevocation` — baseline and extended deployment |
| §3 | `WitUpdate` — device-side epoch update |
| §4 | Manager `BatchAdd` and `BatchDel` vs batch size |
| §5 | DLT / Relay gate check and epoch log fetch |
| §6 | Wire sizes — witness, ΔZ, BF_Δ, epoch packages |

## Setup

**Three ways to get the source onto your Colab runtime** (choose one):

| # | Method | When to use |
|---|--------|-------------|
| A | `git clone` (default in setup cell) | Once the repo is public |
| B | Upload `attest_src.zip` | Before repo is public — run `sh benchmarks/make_colab_zip.sh` locally first |
| C | Google Drive mount | If you have the folder synced to Drive |

**If you already have the zip:** just run the setup cell — it will prompt you to upload it, then handle everything automatically.


In [ ]:
import sys, os

# ── Detect environment ────────────────────────────────────────────────────────
try:
    import google.colab; IN_COLAB = True
except ImportError:
    IN_COLAB = False

_src_ok = os.path.isdir('attest') and os.path.isdir('compass')

if IN_COLAB and not _src_ok:
    # ── Option A: GitHub clone (uncomment once repo is public) ────────────────
    # !git clone https://github.com/khsjds/ATTEST.git
    # os.chdir('ATTEST')

    # ── Option B: upload attest_src.zip ───────────────────────────────────────
    # Run locally first:  cd 4_implementation && sh benchmarks/make_colab_zip.sh
    # That creates attest_src.zip — upload it when prompted below.
    import zipfile
    from google.colab import files
    print('Upload attest_src.zip (create it locally: sh benchmarks/make_colab_zip.sh)')
    uploaded = files.upload()              # opens file picker
    zip_name = next(iter(uploaded))
    with zipfile.ZipFile(zip_name, 'r') as zf:
        zf.extractall('.')
    os.remove(zip_name)

    # ── Option C: Google Drive ────────────────────────────────────────────────
    # from google.colab import drive; drive.mount('/content/drive')
    # os.chdir('/content/drive/MyDrive/ATTEST/4_implementation')

# ── Install / verify dependencies ─────────────────────────────────────────────
!pip install numpy --quiet
sys.path.insert(0, '.')

try:
    import attest, compass
    print(f'attest {attest.__version__}  |  compass OK  — setup complete.')
    if not IN_COLAB:
        print('(Running locally — make sure you are in 4_implementation/)')
except ImportError as e:
    print(f'Import failed: {e}')
    print('If running locally: cd to 4_implementation/ before launching Jupyter.')
    print('On Colab: re-run this cell and upload attest_src.zip when prompted.')


## Parameter Set Selection

Choose which parameter set to benchmark:

| | **set1** | **set2** (paper) |
|---|---|---|
| Ring dim N | 512 | **1024** |
| Modulus q | ~2²⁷·⁶ | **~2³²** |
| Witness size | 4.2 KB | **8.2 KB** |
| ΔZ wire size | 2.0 KB | **4.0 KB** |
| Verify (Colab) | ~8 ms | **~80 ms** |
| WitUpdate | ~0.4 ms | **~0.7 ms** |
| Runtime (this nb) | **~1 min** | ~8 min |

**Recommendation:** use `set2` to reproduce paper numbers.  
Use `set1` for a quick functional check.

In [ ]:
# ── Choose parameter set ───────────────────────────────────────────────────────
PARAM_SET = "set2"   # <── change to "set1" for a quick run

# ── Benchmark configuration ───────────────────────────────────────────────────
K_POOL   = 20     # initial accumulator pool size (keep small for speed)
N_VERIFY = 10     # verification repetitions  (~80ms × 10 = 14s for set2)
N_FAST   = 200    # fast-op repetitions       (μs-range ops)

# ── Imports ───────────────────────────────────────────────────────────────────
import time, statistics
import numpy as np

from attest import (
    SET1, SET2, PARAM_SETS,
    VC, H_tag, BloomFilter,
    manager_setup, batch_add, batch_del,
    dlt_init, dlt_publish, dlt_gate_check, dlt_fetch_updates,
    relay_init, relay_sync, relay_gate_check, relay_fetch_updates,
    wit_update, verify_with_revocation,
)
from attest.params import BF_FP_RATE

ps = PARAM_SETS[PARAM_SET]
print(ps)
print(f"K_pool={K_POOL}, N_verify={N_VERIFY}, N_fast={N_FAST}")

In [ ]:
# ── Helper functions ──────────────────────────────────────────────────────────
def make_vcs(n, issuer=b'bench'):
    return [VC(issuer_id=issuer, cred_id=f'cred_{i}'.encode(),
               holder_pk=(f'pk_{i}'.encode() * 4)[:32], issued_epoch=0)
            for i in range(n)]

def timeit(fn, n_runs=10):
    """Return (mean_ms, std_ms)."""
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        fn()
        times.append((time.perf_counter() - t0) * 1000)
    return statistics.mean(times), (statistics.stdev(times) if n_runs > 1 else 0.0)

def row(label, mean_ms, std_ms=0, unit='ms', note=''):
    std_s = f' ± {std_ms:.3f}' if std_ms > 0 else ''
    note_s = f'  ({note})' if note else ''
    print(f'  {label:<46} {mean_ms:>9.3f} {unit}{std_s}{note_s}')

print('Helpers ready.')

## Initialise Manager + DLT

*(One-time setup — not benchmarked)*

In [ ]:
print(f'Building initial accumulator ({K_POOL} members, param_set={PARAM_SET}) …')
t0 = time.perf_counter()

mgr  = manager_setup(K=K_POOL, param_set=PARAM_SET)
dlt  = dlt_init(mgr)
vcs  = make_vcs(K_POOL)
mgr, witnesses, pkg = batch_add(mgr, vcs)
dlt  = dlt_publish(dlt, pkg)
relay = relay_init(dlt)

setup_s = time.perf_counter() - t0
c = mgr.compass
print(f'Done in {setup_s:.1f} s  ({setup_s*1000/K_POOL:.0f} ms/member)')
print(f'N={c.N}, q={c.q}, κ={c.kappa}, σ={c.sigma}, t={c.t}')

## §1 — Component Micro-benchmarks

Individual operations that compose the higher-level algorithms.

In [ ]:
tag = H_tag(vcs[0])
bf  = BloomFilter(K=K_POOL, p=BF_FP_RATE)
bf2 = BloomFilter(K=K_POOL, p=BF_FP_RATE); bf2.insert(tag)

Zi, zi, ci1, ci2 = c.deserialize_witness(witnesses[0].data)
dZ = pkg.delta_Z
Z  = mgr.Z

m_ins, s_ins   = timeit(lambda: bf.insert(tag),                                    N_FAST)
m_qry, s_qry   = timeit(lambda: bf.query(tag),                                     N_FAST)
m_mrg, s_mrg   = timeit(lambda: BloomFilter(K=K_POOL, p=BF_FP_RATE,
                                             _bits=bytearray(bf.serialize())).merge(bf2), N_FAST)
m_radd, s_radd = timeit(lambda: (Zi + dZ) % c.q,                                  N_FAST)
m_pf, s_pf     = timeit(lambda: c.PF.F(Z),                                        N_FAST)

print(f'\n  {"Operation":<46} {"Mean":>9}  ±std')
row('BF.insert (1 tag)',                                    m_ins,  s_ins)
row('BF.query  (1 tag)',                                    m_qry,  s_qry)
row(f'BF.merge  (full {bf.size_bytes()} B bitmap)',         m_mrg,  s_mrg)
row('Ring addition  Z_i + ΔZ mod q',                        m_radd, s_radd)
row(f'PartialFourier F_Ω(Z)  (N={c.N}, t={c.t})',           m_pf,   s_pf)

## §2 — VerifyWithRevocation

Two-phase verification:
- **Baseline**: accumulator check only (`bf_local=None`)
- **Extended**: BF gate (Phase 1) + accumulator (Phase 2)

In [ ]:
# Revoke last device, update device[0]'s witness to current epoch
mgr2, del_pkg = batch_del(mgr, [vcs[-1]])
dlt2 = dlt_publish(dlt, del_pkg)

bf_local = BloomFilter(K=K_POOL, p=BF_FP_RATE)
w0 = witnesses[0]
for p in dlt_fetch_updates(dlt2, from_epoch=1, to_epoch=dlt2.epoch):
    w0, bf_local = wit_update(c, p, w0, bf_local, vcs[0])

print(f'Benchmarking VerifyWithRevocation ({N_VERIFY} runs each) …')
m_vb, s_vb = timeit(lambda: verify_with_revocation(c, dlt2.Acc, vcs[0], w0, None),     N_VERIFY)
m_ve, s_ve = timeit(lambda: verify_with_revocation(c, dlt2.Acc, vcs[0], w0, bf_local), N_VERIFY)
m_bf, s_bf = timeit(lambda: bf_local.query(H_tag(vcs[0])),                              N_FAST)

print(f'\n  {"Operation":<46} {"Mean":>9}  ±std')
row('Phase 1: BF.query (tag not in BF)',         m_bf,  s_bf)
row('Phase 2: COMPASS.Verify (accumulator)',     m_vb,  s_vb,  note='baseline')
row('VerifyWithRevocation — baseline',           m_vb,  s_vb,  note='bf_local=None')
row('VerifyWithRevocation — extended',           m_ve,  s_ve,  note='BF gate + accumulator')
print(f'\n  BF gate overhead: +{m_ve-m_vb:.3f} ms  ({(m_ve-m_vb)/m_vb*100:.1f}% of verify time)')

## §3 — WitUpdate

Device-side epoch update:
- **Baseline**: ring addition only `Z_i* = Z_i + ΔZ`
- **Extended (add epoch)**: ring addition, no BF_Δ to merge
- **Extended (revoke epoch)**: ring addition + BF merge

In [ ]:
# Prepare a fresh add-epoch package
extra_vcs = make_vcs(1, issuer=b'extra')
_, _, add_pkg = batch_add(mgr2, extra_vcs)

bf_dev = BloomFilter(K=K_POOL, p=BF_FP_RATE)

m_wu_b, s_wu_b = timeit(
    lambda: wit_update(c, add_pkg, witnesses[0], None, vcs[0]), N_FAST)

m_wu_e, s_wu_e = timeit(
    lambda: wit_update(c, add_pkg, witnesses[0],
                       BloomFilter(K=K_POOL, p=BF_FP_RATE,
                                   _bits=bytearray(bf_dev.serialize())),
                       vcs[0]), N_FAST)

m_wu_r, s_wu_r = timeit(
    lambda: wit_update(c, del_pkg, witnesses[0],
                       BloomFilter(K=K_POOL, p=BF_FP_RATE,
                                   _bits=bytearray(bf_dev.serialize())),
                       vcs[0]), N_FAST)

print(f'\n  {"Operation":<46} {"Mean":>9}  ±std')
row('WitUpdate — baseline (ring add only)',               m_wu_b, s_wu_b)
row('WitUpdate — extended, add epoch',                   m_wu_e, s_wu_e, note='no BF_Δ')
row('WitUpdate — extended, revoke epoch',                m_wu_r, s_wu_r, note='ring add + BF merge')

## §4 — Manager EpochUpdate vs Batch Size

- `BatchAdd`: uses rejection-sampling (probabilistic); cost is O(m) with ~constant factor
- `BatchDel`: deterministic ring subtractions + BF inserts; essentially constant cost

In [ ]:
ADD_SIZES = [1, 5, 10]
DEL_SIZES = [1, 5, 10]
N_ADD_RUNS = 3
N_DEL_RUNS = 10

print('  BatchAdd (manager side — rejection sampling)')
print(f'  {"m (additions)":<22} {"Mean":>9}  ±std')
add_times = {}
for m_size in ADD_SIZES:
    new_vcs = make_vcs(m_size, issuer=f'add_{m_size}'.encode())
    mean_t, std_t = timeit(lambda: batch_add(mgr, new_vcs), N_ADD_RUNS)
    add_times[m_size] = mean_t
    print(f'  m={m_size:<20} {mean_t:>9.1f} ms  ± {std_t:.1f}  ({mean_t/m_size:.1f} ms/member)')

print()
print('  BatchDel (manager side — deterministic subtraction)')
print(f'  {"ℓ (revocations)":<22} {"Mean":>9}  ±std')
del_times = {}
for l_size in DEL_SIZES:
    del_vcs = vcs[:l_size]
    mean_t, std_t = timeit(lambda: batch_del(mgr, del_vcs), N_DEL_RUNS)
    del_times[l_size] = mean_t
    print(f'  ℓ={l_size:<20} {mean_t:>9.3f} ms  ± {std_t:.3f}  ({mean_t/l_size:.3f} ms/revocation)')

## §5 — DLT and Relay Operations

- **Gate check**: DLT or relay queries BF before serving ΔZ to a device
- **Epoch log fetch**: device retrieves missed epoch packages for catch-up

In [ ]:
relay2 = relay_init(dlt2)

# Extend log for multi-epoch fetch benchmark
dlt_long, mgr_long = dlt2, mgr2
for i in range(8):
    extra = make_vcs(1, issuer=f'long_{i}'.encode())
    mgr_long, _, p_tmp = batch_add(mgr_long, extra)
    dlt_long = dlt_publish(dlt_long, p_tmp)

m_gc,  s_gc  = timeit(lambda: dlt_gate_check(dlt2, vcs[0]),  N_FAST)  # not revoked
m_gcr, s_gcr = timeit(lambda: dlt_gate_check(dlt2, vcs[-1]), N_FAST)  # revoked
m_pub, s_pub = timeit(lambda: dlt_publish(dlt, del_pkg),     N_FAST)
m_f1,  s_f1  = timeit(lambda: dlt_fetch_updates(dlt2,     from_epoch=1, to_epoch=2), N_FAST)
m_f5,  s_f5  = timeit(lambda: dlt_fetch_updates(dlt_long, from_epoch=2, to_epoch=7), N_FAST)
m_f8,  s_f8  = timeit(lambda: dlt_fetch_updates(dlt_long,
                                                  from_epoch=dlt_long.epoch-8,
                                                  to_epoch=dlt_long.epoch), N_FAST)

print(f'\n  {"Operation":<46} {"Mean":>9}  ±std')
row('dlt_gate_check  (not revoked → allow)',  m_gc,  s_gc)
row('dlt_gate_check  (revoked → deny)',       m_gcr, s_gcr)
row('dlt_publish     (BF merge + log write)', m_pub, s_pub)
row('dlt_fetch_updates  1 missed epoch',      m_f1,  s_f1)
row('dlt_fetch_updates  5 missed epochs',     m_f5,  s_f5)
row('dlt_fetch_updates  8 missed epochs',     m_f8,  s_f8)

## §6 — Wire Sizes

All sizes use serialised representation (4 bytes/coefficient for set2 q≈2³²).
The epoch broadcast does **not** include Acc (stored on-chain, fetched separately).

In [ ]:
from compass.utils import bytes_per_q as _bpq
bpq = _bpq(c.q)

w_size       = len(witnesses[0].data)
dz_size      = len(pkg.delta_Z) * bpq
acc_size     = len(dlt.Acc) * bpq
sig_size     = len(pkg.sig)
bf_k20_size  = BloomFilter(K=K_POOL,  p=BF_FP_RATE).size_bytes()
bf_k10k_size = BloomFilter(K=10_000,  p=BF_FP_RATE).size_bytes()
add_bcast    = dz_size + sig_size            # ΔZ + sig (Acc is on-chain)
del_bcast    = dz_size + bf_k10k_size + sig_size

items = [
    ('Witness (Z_i, z_i, c1, c2)',                w_size),
    (f'ΔZ (ring element, N={c.N})',               dz_size),
    (f'Acc digest (t={c.t} coeffs)',              acc_size),
    (f'BF bitmap (K={K_POOL}, p=1%)',             bf_k20_size),
    ('BF bitmap (K=10000, p=1%)',                 bf_k10k_size),
    ('Epoch broadcast: add-only (ΔZ + sig)',      add_bcast),
    ('Epoch broadcast: revoke (ΔZ+BF_Δ+sig)',     del_bcast),
]

print(f'  {"Component":<46} {"Bytes":>7}  KB')
for lbl, sz in items:
    print(f'  {lbl:<46} {sz:>7}  ({sz/1024:.1f} KB)')

## Summary — Paper §5 Table

In [ ]:
import platform
hw = platform.processor() or platform.machine()
py = platform.python_version()

print(f'ATTEST — Performance Summary')
print(f'param_set={PARAM_SET}  N={c.N}  q={c.q}')
print(f'Python {py}  /  {hw}')
print()
print('  Timing')
print(f'  {"Operation":<44} {"Time"}')
print('  ' + '-'*60)
for lbl, val, unit in [
    ('VerifyWithRevocation (baseline)',  f'{m_vb:.1f} ± {s_vb:.1f}', 'ms'),
    ('VerifyWithRevocation (extended)', f'{m_ve:.1f} ± {s_ve:.1f}', 'ms'),
    ('WitUpdate (baseline)',             f'{m_wu_b:.3f} ± {s_wu_b:.3f}', 'ms'),
    ('WitUpdate (extended, revoke)',     f'{m_wu_r:.3f} ± {s_wu_r:.3f}', 'ms'),
    ('DLT gate check',                  f'{m_gc*1000:.2f} ± {s_gc*1000:.2f}', 'μs'),
    ('BatchAdd per member (m=1)',        f'{add_times[1]:.1f}', 'ms'),
    ('BatchDel (ℓ=1)',                   f'{del_times[1]*1000:.1f}', 'μs'),
]:
    print(f'  {lbl:<44} {val} {unit}')

print()
print('  Wire sizes')
print(f'  {"Component":<44} {"Size"}')
print('  ' + '-'*60)
for lbl, sz in [
    ('Witness',                         f'{w_size} B ({w_size/1024:.1f} KB)'),
    ('ΔZ (epoch broadcast element)',    f'{dz_size} B ({dz_size/1024:.1f} KB)'),
    ('BF_Δ (K=10000, p=1%)',            f'{bf_k10k_size} B ({bf_k10k_size/1024:.1f} KB)'),
    ('Full add-only epoch broadcast',   f'{add_bcast} B ({add_bcast/1024:.1f} KB)'),
    ('Full revocation epoch broadcast', f'{del_bcast} B ({del_bcast/1024:.1f} KB)'),
]:
    print(f'  {lbl:<44} {sz}')